# Fine-Tune on Buildings 900K

The TTM way first (each time serie cut into train/val/test), with a domain-shit eval way second

In [1]:
import tempfile
import math
import os
import pandas as pd

from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from transformers import EarlyStoppingCallback, Trainer, TrainingArguments, set_seed
from transformers.integrations import INTEGRATION_TO_CALLBACK

from tsfm_public import TimeSeriesPreprocessor, TrackingCallback, count_parameters, get_datasets
from tsfm_public.toolkit.get_model import get_model
from tsfm_public.toolkit.lr_finder import optimal_lr_finder
from tsfm_public.toolkit.visualization import plot_predictions

import datasets
import numpy as np

from gluonts.model.forecast import SampleForecast
from gluonts.model.evaluation import evaluate_forecasts
from gluonts.ev.metrics import MASE, MeanWeightedSumQuantileLoss
from gluonts.dataset.split import split

/home/joffreyma/miniforge3/envs/chronos_env/lib/python3.11/site-packages/gluonts/json.py:102: UserWarning: Using `json`-module for json-handling. Consider installing one of `orjson`, `ujson` to speed up serialization and deserialization.
  warnings.warn(


In [2]:
import warnings


# Suppress all warnings
warnings.filterwarnings("ignore")

In [ ]:
TTM_MODEL_PATH = "ibm-granite/granite-timeseries-ttm-r2"
# Set seed for reproducibility
SEED = 41
set_seed(SEED)

# Results dir
OUT_DIR = "ttm_finetuned_models/"
dataset_name = "buildings_900K"

context_length=512
forecast_length=64
train_batch_size=1024
eval_batch_size=100000

pd_batch_size = 1024 # 5000

# Dataset
timestamp_column = "timestamp"
id_columns = ["item_id"]  # mention the ids that uniquely identify a time-series.

target_columns = ["target"]
split_config = {
    "train": [
        0,
        4000,# 8761-forecast_length-100,
    ],
    "valid": [
        4000,# 8761-forecast_length-100,
        8761-forecast_length,
    ],
    "test": [
        8761-forecast_length, # the context length is included by the splitter
        8761,
    ],
}
# Understanding the split config -- slides

column_specifiers = {
    "timestamp_column": timestamp_column,
    "id_columns": id_columns,
    "target_columns": target_columns,
    "control_columns": [],
}


In [4]:
files = ["../../../chronos-forecasting/data/buildings_900K_chronos_split_ttm/domain_shift/data-00000-of-00001.arrow"]
ds = datasets.load_dataset(
    "arrow", data_files={'train': files}, split="train"
)

In [5]:
# data = ds.with_format("pandas").map(lambda batch: batch.explode(["target", "timestamp"], ignore_index=True), batched=True, batch_size=pd_batch_size)

# Fine-tuning code

In [6]:
dataset_name=dataset_name
context_length=context_length
forecast_length=forecast_length
fewshot_percent=1
learning_rate=0.001
freeze_backbone=True
num_epochs=1
save_dir=OUT_DIR
loss="mse"
quantile=0.5

In [7]:
out_dir = os.path.join(save_dir, dataset_name)

print("-" * 20, f"Running few-shot {fewshot_percent}%", "-" * 20)

# Data prep: Get dataset

tsp = TimeSeriesPreprocessor(
    **column_specifiers,
    context_length=context_length,
    prediction_length=forecast_length,
    scaling=True,
    encode_categorical=False,
    scaler_type="standard",
)

# change head dropout to 0.7 for ett datasets
finetune_forecast_model = get_model(
    TTM_MODEL_PATH,
    context_length=context_length,
    prediction_length=forecast_length,
    freq_prefix_tuning=False,
    freq=None,
    prefer_l1_loss=False,
    prefer_longer_context=True,
    # Can also provide TTM Config args
    loss=loss,
    quantile=quantile,
)

INFO:p-62039:t-132482641962112:get_model.py:get_model:Loading model from: ibm-granite/granite-timeseries-ttm-r2


-------------------- Running few-shot 1% --------------------


INFO:p-62039:t-132482641962112:get_model.py:get_model:Model loaded successfully from ibm-granite/granite-timeseries-ttm-r2, revision = main.
INFO:p-62039:t-132482641962112:get_model.py:get_model:[TTM] context_length = 512, prediction_length = 96


In [8]:
if freeze_backbone:
    print(
        "Number of params before freezing backbone",
        count_parameters(finetune_forecast_model),
    )

    # Freeze the backbone of the model
    for param in finetune_forecast_model.backbone.parameters():
        param.requires_grad = False

    # Count params
    print(
        "Number of params after freezing the backbone",
        count_parameters(finetune_forecast_model),
    )

Number of params before freezing backbone 805280
Number of params after freezing the backbone 289696


In [9]:
# Optimizer and scheduler
optimizer = AdamW(finetune_forecast_model.parameters(), lr=learning_rate)
scheduler = OneCycleLR(
    optimizer,
    learning_rate,
    epochs=num_epochs,
    steps_per_epoch=math.ceil(ds.num_rows / (train_batch_size)),
)

In [10]:
finetune_forecast_args = TrainingArguments(
    output_dir=os.path.join(out_dir, "output"),
    overwrite_output_dir=True,
    learning_rate=learning_rate,
    num_train_epochs=1,
    do_eval=True,
    eval_strategy="epoch",
    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=eval_batch_size,
    dataloader_num_workers=8,
    report_to="none",
    save_strategy="epoch",
    logging_strategy="epoch",
    save_total_limit=1,
    logging_dir=os.path.join(out_dir, "logs"),  # Make sure to specify a logging directory
    load_best_model_at_end=True,  # Load the best model when training ends
    metric_for_best_model="eval_loss",  # Metric to monitor for early stopping
    greater_is_better=False,  # For loss
    seed=SEED,
)

In [11]:
for _ in range(num_epochs):
    data = ds.to_pandas(batch_size=pd_batch_size, batched=True)
    for sub_data in data:
        sub_data_exploded = sub_data.explode(["target", "timestamp"], ignore_index=True)
        #stride = context_length / 2

        dset_train, dset_val, dset_test = get_datasets(
            tsp,
            sub_data_exploded,
            split_config,
            fewshot_fraction=fewshot_percent / 100,
            fewshot_location="last",
            use_frequency_token=finetune_forecast_model.config.resolution_prefix_tuning,
        )
    
        del sub_data_exploded

        print(f"Using learning rate = {learning_rate}")
        
        tracking_callback = TrackingCallback()

        finetune_forecast_trainer = Trainer(
            model=finetune_forecast_model,
            args=finetune_forecast_args,
            train_dataset=dset_train,
            eval_dataset=dset_val,
            callbacks=[tracking_callback],
            optimizers=(optimizer, scheduler),
        )
        finetune_forecast_trainer.remove_callback(INTEGRATION_TO_CALLBACK["codecarbon"])

        # Fine tune
        finetune_forecast_trainer.train()

        # Evaluation
        print("+" * 20, f"Test MSE after few-shot {fewshot_percent}% fine-tuning", "+" * 20)

        finetune_forecast_trainer.model.loss = "mse"  # fixing metric to mse for evaluation

        fewshot_output = finetune_forecast_trainer.evaluate(dset_test)
        print(fewshot_output)
        print("+" * 60)


Using learning rate = 0.001


Epoch,Training Loss,Validation Loss
1,0.153200,0.198746


[TrackingCallback] Mean Epoch Time = 0.6400167942047119 seconds, Total Train Time = 72.28503084182739
++++++++++++++++++++ Test MSE after few-shot 1% fine-tuning ++++++++++++++++++++


{'eval_loss': 0.3408213257789612, 'eval_runtime': 0.3403, 'eval_samples_per_second': 3009.221, 'eval_steps_per_second': 2.939, 'epoch': 1.0}
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Using learning rate = 0.001


Epoch,Training Loss,Validation Loss
1,0.131300,0.186668


...

Epoch,Training Loss,Validation Loss
1,0.063500,0.210318


[TrackingCallback] Mean Epoch Time = 0.64202880859375 seconds, Total Train Time = 74.3097312450409
++++++++++++++++++++ Test MSE after few-shot 1% fine-tuning ++++++++++++++++++++


{'eval_loss': 0.42740964889526367, 'eval_runtime': 0.562, 'eval_samples_per_second': 1822.026, 'eval_steps_per_second': 1.779, 'epoch': 1.0}
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Using learning rate = 0.001


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
""" # get predictions

predictions_dict = finetune_forecast_trainer.predict(dset_test)

predictions_np = predictions_dict.predictions[0]

print(predictions_np.shape) """

' # get predictions\n\npredictions_dict = finetune_forecast_trainer.predict(dset_test)\n\npredictions_np = predictions_dict.predictions[0]\n\nprint(predictions_np.shape) '

In [ ]:
""" # plot
plot_predictions(
    model=finetune_forecast_trainer.model,
    dset=dset_test,
    plot_dir=os.path.join(OUT_DIR, dataset_name),
    plot_prefix="test_fewshot",
    indices=[0, 1],
    channel=0,
) """

' # plot\nplot_predictions(\n    model=finetune_forecast_trainer.model,\n    dset=dset_test,\n    plot_dir=os.path.join(OUT_DIR, dataset_name),\n    plot_prefix="test_fewshot",\n    indices=[0, 1],\n    channel=0,\n) '